In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import HTML
from tqdm import tqdm
from astropy.visualization import simple_norm
from astropy.stats import sigma_clipped_stats
from astropy.time import Time
from tess_asteroids.utils import animate_cube, plot_img_aperture

matplotlib.rcParams["animation.embed_limit"] = 2**128

In [6]:
data = np.load(f"../../data/2025/3i_s0092_docu_animation_images.npz")

times = data["time"]
flux_cube = data["flux_cube"]
flux_shifted_cube = data["flux_shifted_cube"]
flux_stacked_cube = data["flux_stacked_cube"]
cumulative_stack_raw = data["cumulative_stack_raw"]
cumulative_stack_shifted = data["cumulative_stack_shifted"]
paper_stacks = data["paper_stacks"]
paper_times = data["paper_times"]

In [7]:
TIMES = Time(times + 2457000, format="jd", scale="tdb")

TIMES[0].iso, TIMES[-1].iso

('2025-05-07 11:26:25.301', '2025-06-02 00:56:07.346')

In [8]:
from typing import Optional, Tuple, Union
from astropy.visualization import simple_norm
from matplotlib import animation, colors, patches

def plot_img_aperture(
    img: np.ndarray,
    aperture_mask: Optional[np.ndarray] = None,
    cbar: bool = True,
    ax: Optional[plt.Axes] = None,
    corner: Tuple[int, int] = (0, 0),
    marker: Optional[Tuple[float, float]] = None,
    title: str = "",
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    cnorm: Optional[colors.Normalize] = None,
):
    """
    Plots an image with an optional aperture mask.

    This function displays an image, optionally overlaying an aperture mask, and
    provides several customization options such as color scaling, title, and axis control.

    Parameters
    ----------
    img : 2D array
        The image data to be plotted, typically a 2D array or matrix representing pixel values.
    aperture_mask : 2D array, default=None
        A binary mask (same shape as `img`) indicating the aperture region to be overlaid on the image.
    cbar : bool, default=True
        Whether to display a color bar alongside the plot.
    ax : matplotlib.axes.Axes, default=None
        The axes object where the plot will be drawn. If not provided, a new axes will be created.
    corner : list of two ints, default=[0, 0]
        The (row, column) coordinates of the lower left corner of the image.
    marker : tuple of float, default=None
        The (row, column) coordinates at which to plot a marker in the figure.
        This can be used to plot the position of the moving object.
    title : str, default=""
        Title of the plot. If None, no title will be shown.
    vmin : float, optional, default=None
        Minimum value for color scale. If None, the 3%-percentile is used.
    vmax : float, optional, default=None
        Maximum value for color scale. If None, the 97%-percentile is used.
    cnorm : optional, default=None
        Color matplotlib normalization object (e.g. astropy.visualization.simple_norm). If provided,
        then `vmax` and `vmin` are not used.

    Returns
    -------
    ax : matplotlib.axes.Axes
        The axes object containing the plot.
    """

    # Initialise ax
    if ax is None:
        _, ax = plt.subplots()

    # Define the x and y axis tick labels when using plt.imshow() using `corner`
    # and the image shape
    extent = (
        corner[1] - 0.5,
        corner[1] + img.shape[1] - 0.5,
        corner[0] - 0.5,
        corner[0] + img.shape[0] - 0.5,
    )

    # Define vmin and vmax
    if vmin is None and vmax is None and cnorm is None:
        vmin, vmax = np.nanpercentile(img.ravel(), [3, 97])

    # Plot image, colorbar and marker
    im = ax.imshow(
        img,
        cmap="Greys_r",
        vmin=vmin,
        vmax=vmax,
        norm=cnorm,
        rasterized=False,
        origin="lower",
        extent=extent,
    )
    if cbar:
        plt.colorbar(im, location="right", shrink=0.8, label="Flux [e-/s]")
    if marker is not None:
        ax.scatter(marker[1], marker[0], marker="x", c="deeppink", alpha=1, s=50)

    ax.set_aspect("equal", "box")
    ax.set_title(title)

    # ax.set_xlabel("Pixel Column")
    # ax.set_ylabel("Pixel Row")
    ax.set_xticks([])
    ax.set_yticks([])

    return ax


def animate_cube(
    cube: np.ndarray,
    aperture_mask: Optional[np.ndarray] = None,
    corner: Union[Tuple, np.ndarray] = (0, 0),
    ephemeris: Optional[np.ndarray] = None,
    cadenceno: Optional[np.ndarray] = None,
    time: Optional[np.ndarray] = None,
    interval: int = 200,
    repeat_delay: int = 1000,
    step: int = 1,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    cnorm: bool = False,
    suptitle: str = "",
    date_str: str = "",
    date_units: str = "",
):
    """
    Creates an animated visualization of a 3D image cube, with an optional aperture mask and
    other customization options.

    This function animates the slices of a 3D image cube, optionally overlaying an aperture mask,
    and provides controls for animation speed, title, and tracking information.

    Parameters
    ----------
    cube : 3D array
        A 3D array representing the image cube (e.g., a stack of 2D images over time).
    aperture_mask : 2D or 3D array, optional
        A binary mask (same shape or a 2D slice of `cube`) to overlay on each frame of the animation.
        If a 2D mask is passed, it will be repeated for all times.
    corner : list of two ints or 2D array, default=[0, 0]
        The (row, column) coordinates of the lower left corner of the image.
    ephemeris : 2D array, optional, default=None
        A 2D array of object positions (row, column) to be displayed on the plot.
        For proper display of object position, if `corner` is [0, 0] then `ephemeris` needs to be relative to `corner`.
        If `corner` is provided, `ephemeris` needs to be absolute.
        If None, no tracking information is shown.
    cadenceno : int, optional, default=None
        The cadence number of the frames, used for information display.
    time : array-like, optional, default=None
        Array of time values corresponding to the slices in the cube.
    interval : int, default=200
        The time interval (in milliseconds) between each frame of the animation.
    repeat_delay : int, default=1000
        The time delay (in milliseconds) before the animation restarts once it finishes.
    step : int
        Spacing between frames, i.e. plot every nth frame.
    vmin : float, optional, default=None
        Minimum value for color scale. If None, the 3%-percentile is used.
    vmax : float, optional, default=None
        Maximum value for color scale. If None, the 97%-percentile is used.
    cnorm : optional, default=False
        Whether to use asinh color normalization (from astropy.visualization.simple_norm).
        This can be useful for cases when the moving object is too faint compared to other
        features in the background. If provided, then `vmax` and `vmin` are not used.
    suptitle : str, optional, default=""
        A string to be used as the super title of the animation.
        It can be used to provide additional context or information about the animated data,
        for example the target name or observing sector/camera/ccd.

    Returns
    -------
    ani : matplotlib.animation.FuncAnimation
        The animation object that can be displayed or saved.
    """

    # Initialise figure and set title
    fig, ax = plt.subplots()
    fig.suptitle(suptitle)

    if aperture_mask is None:
        aperture_mask = np.repeat([None], len(cube), axis=0)
    # If aperture_mask is 2D, repeat for all times.
    elif aperture_mask.shape == cube.shape[1:]:
        aperture_mask = np.repeat([aperture_mask], len(cube), axis=0)

    if ephemeris is None:
        ephemeris = np.repeat([None], len(cube), axis=0)
    if cadenceno is None:
        cadenceno = np.repeat([None], len(cube), axis=0)
    if time is None:
        time = np.repeat([None], len(cube), axis=0)

    if cnorm:
        norm = simple_norm(cube.ravel(), "asinh", percent=98)
    elif vmin is None and vmax is None:
        # Ignore warning that arises from all NaN values.
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore", message="All-NaN slice encountered", category=RuntimeWarning
            )
            vmin, vmax = np.nanpercentile(cube, [3, 97])

    # If corner is list of two ints, repeat for all times.
    if len(corner) == 2:
        corner = np.repeat([corner], len(cube), axis=0)

    # Plot first image in cube.
    nt = 0
    ax = plot_img_aperture(
        cube[nt],
        aperture_mask=aperture_mask[nt],
        cbar=True,
        ax=ax,
        corner=corner[nt],
        marker=ephemeris[nt],
        title=f"{date_str} {time[nt]} {date_units}",
        vmin=vmin if not cnorm else None,
        vmax=vmax if not cnorm else None,
        cnorm=norm if cnorm else None,
    )

    # Define function for animation
    def animate(nt):
        ax.clear()
        _ = plot_img_aperture(
            cube[nt],
            aperture_mask=aperture_mask[nt],
            cbar=False,
            ax=ax,
            corner=corner[nt],
            marker=ephemeris[nt],
            title=f"{date_str} {time[nt]} {date_units}",
            vmin=vmin if not cnorm else None,
            vmax=vmax if not cnorm else None,
            cnorm=norm if cnorm else None,
        )

        return ()

    # Prevent second figure from showing up in interactive mode
    plt.close(ax.figure)  # type: ignore

    # Create the animation
    ani = animation.FuncAnimation(
        fig,
        animate,
        frames=range(0, len(cube), step),
        interval=interval,
        blit=True,
        repeat_delay=repeat_delay,
        repeat=True,
    )

    return ani

def tqdm_callback(current_frame, total_frames_val):
    # 'total_frames_val' might be None if not determined, so use the known total_frames
    if not hasattr(tqdm_callback, 'pbar'):
        # Initialize the progress bar on the first call
        tqdm_callback.pbar = tqdm(total=total_frames_val or total_frames, desc="Saving Animation")
    
    # Update the progress bar
    tqdm_callback.pbar.update(1)
    
    # Close the progress bar when finished
    if current_frame == (total_frames_val or total_frames) - 1:
        tqdm_callback.pbar.close()

In [9]:
cumulative_stack_shifted.shape

(9711, 21, 21)

In [10]:
# skip first 10 frames
ns = 10
nr = 1000
# hold last frame longer
flux_to_animate = cumulative_stack_shifted[ns:, 1:-1, 1:-1]
last_frame = flux_to_animate[-1][None, :, :]
last_frame = np.repeat(last_frame, nr, axis=0)
flux_to_animate = np.concatenate([flux_to_animate, last_frame], axis=0)

time_to_animate = times[ns:] - times[0]
last_time = time_to_animate[-1]
last_time = np.repeat(last_time, nr, axis=0)
time_to_animate = np.concatenate([time_to_animate, last_time], axis=0)

flux_to_animate.shape, time_to_animate.shape

((10701, 19, 19), (10701,))

In [11]:
ani = animate_cube(
    flux_to_animate,
    step=25,
    time=np.array([f"{x:.4f}" for x in time_to_animate]),
    suptitle=f"NASA TESS - 3I/ATLAS (2025 May 7 $-$ June 02)",
    vmin=-0.1,
    vmax=0.3,
    interval=50,
    date_str="Cumulative Observations",
    date_units="[d]",
)
ani.save(
    f"../../data/2025/figures/3i_s0092_docu_animation_cumulative.mp4", 
    fps=30, 
    writer="ffmpeg", # ffmpeg
    dpi=300,
    bitrate=-1,
    progress_callback=tqdm_callback,
)
HTML(ani.to_jshtml())

MovieWriter ffmpeg unavailable; using Pillow instead.
Saving Animation: 100%|███████████████████████████████████████████████████████| 429/429 [00:19<00:00, 22.17it/s]


ValueError: unknown file extension: .mp4

In [104]:
P_TIMES = Time(paper_times, format="jd", scale="utc")
P_TIMES = np.array([x[:10] for x in P_TIMES.iso])
P_TIMES

array(['2025-05-08', '2025-05-10', '2025-05-12', '2025-05-14',
       '2025-05-17', '2025-05-21', '2025-05-22', '2025-05-23',
       '2025-05-24', '2025-05-26', '2025-05-27', '2025-05-28',
       '2025-05-30', '2025-05-31', '2025-06-01'], dtype='<U10')

In [107]:
ani = animate_cube(
    paper_stacks[:, 1:-1, 1:-1],
    step=1,
    # corner=mtpf.corner[time_mask],
    # ephemeris=np.array([Y, X]).T,
    # cadenceno=mtpf.cadence_number[time_mask],
    time=P_TIMES,
    suptitle=f"NASA TESS - 3I/ATLAS (2025 May 7 $-$ June 02)",
    vmin=-0.2,
    vmax=0.4,
    interval=200,
    date_str="Cumulative Observations",
)
ani.save(
    f"../../data/2025/figures/3i_s0092_docu_animation_paper.gif", 
    fps=2, 
    writer="pillow", # ffmpeg
    dpi=300,
    bitrate=-1,
    progress_callback=tqdm_callback,
)
HTML(ani.to_jshtml())